# Random Forest Training and Feature Ablations

Train RF on the full feature set and leave-one-group-out ablations. RF receives unscaled numeric inputs.


**Feature-set ablations:** full features; base features only; full features without balance features; without initiator activity features; and without recipient-risk features.


The six-step delay is the validation period. It is used to choose the fraud threshold, and is not used to fit model parameters.


## 1. Imports and paths

In [1]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

DATA_DIR = r"../results/featured_dataset/featured_144days.pkl"
RESULTS_DIR = r"../results/rf"
RANDOM_STATE = 42


## 2. Shared chronological split and ablation groups

In [2]:
# Sort transactions by time before making chronological splits.
df = pd.read_pickle(DATA_DIR)
df = df.sort_values(["step", "index"]).reset_index(drop=True)

max_step = int(df["step"].max())
train_end = int(max_step * 0.8)
delay = 6
val_end = train_end + delay
test_start = val_end + 1

# Use all rows before the delay to train; use the delay period as validation.
train_df = df[df["step"] <= train_end]
val_df = df[(df["step"] > train_end) & (df["step"] <= val_end)]
test_df = df[df["step"] >= test_start]

print("Train steps:", train_df["step"].min(), "to", train_df["step"].max())
print("Validation (delay) steps:", val_df["step"].min(), "to", val_df["step"].max())
print("Test steps:", test_df["step"].min(), "to", test_df["step"].max())
print("Train rows:", len(train_df), " | Validation rows:", len(val_df), " | Test rows:", len(test_df))

# Separate labels from inputs and remove IDs and time from model features.
y_train = train_df["isFraud"]
y_val = val_df["isFraud"]
y_test = test_df["isFraud"]
drop_cols = ["step", "recipient", "initiator", "index", "isFraud"]
x_train = train_df.drop(columns=drop_cols)
x_val = val_df.drop(columns=drop_cols)
x_test = test_df.drop(columns=drop_cols)

# Group engineered features for the ablation experiments.
balance_features = [
    "amount_balance_ratio", "balance_increase_ratio", "neg_balance_i",
    "balance_error_i", "balance_error_r",
]
initiator_features = [
    "INIT_AVG_AMOUNT_TX_6", "INIT_AMOUNT_DEV_TX_6",
    "INIT_AVG_AMOUNT_TX_12", "INIT_AMOUNT_DEV_TX_12",
    "INIT_AVG_AMOUNT_TX_24", "INIT_AMOUNT_DEV_TX_24",
    "INIT_TX_COUNT_STEP_6", "INIT_TX_COUNT_STEP_12", "INIT_TX_COUNT_STEP_24",
]
recipient_features = [
    "RECIP_RISK_6", "RECIP_TX_COUNT_6", "RECIP_RISK_12",
    "RECIP_TX_COUNT_12", "RECIP_RISK_24", "RECIP_TX_COUNT_24",
]
engineered_features = balance_features + initiator_features + recipient_features
base_features = [col for col in x_train.columns if col not in engineered_features]

feature_sets = {
    "full": list(x_train.columns),
    "base_only": base_features,
    "without_balance": [col for col in x_train.columns if col not in balance_features],
    "without_initiator_activity": [col for col in x_train.columns if col not in initiator_features],
    "without_recipient_risk": [col for col in x_train.columns if col not in recipient_features],
}

print("Feature counts:")
for name, columns in feature_sets.items():
    print(name, len(columns))


Train steps: 0 to 114
Validation (delay) steps: 115 to 120
Test steps: 121 to 143
Train rows: 1361460  | Validation rows: 55663  | Test rows: 303058
Feature counts:
full 30
base_only 10
without_balance 25
without_initiator_activity 21
without_recipient_risk 24


# Train all ablations and save prediction artifacts

## 3. Hyperparameter ablation: min_samples_leaf

Vary only `min_samples_leaf` to test tree regularization while keeping the full feature set, chronological split, class weighting, and all other model settings fixed. Compare validation average precision (AP); use validation results for model/threshold selection and reserve the test period for final evaluation.

In [ ]:
from sklearn.metrics import average_precision_score

# Keep the complete feature set fixed for every configuration.
full_features = list(x_train.columns)
hyperparameter_values = [1, 2, 4, 8, 16]
hyperparameter_results = []
hyperparameter_results_dir = os.path.join(RESULTS_DIR, "hyperparameter_ablation")
os.makedirs(hyperparameter_results_dir, exist_ok=True)

for leaf_size in hyperparameter_values:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=leaf_size,
        max_features="sqrt",
        max_samples=0.6,
        oob_score=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(x_train[full_features], y_train)

    val_scores = model.predict_proba(x_val[full_features])[:, 1]
    artifact_name = f"min_samples_leaf_{leaf_size}"

    joblib.dump(model, os.path.join(hyperparameter_results_dir, f"model_{artifact_name}.pkl"))
    pd.DataFrame({"isFraud": y_val, "score": val_scores}).to_pickle(
        os.path.join(hyperparameter_results_dir, f"{artifact_name}_validation.pkl")
    )
    hyperparameter_results.append({
        "setting": artifact_name,
        "min_samples_leaf": leaf_size,
        "validation_average_precision": average_precision_score(y_val, val_scores),
        "oob_score": model.oob_score_,
    })

results_df = pd.DataFrame(hyperparameter_results).sort_values(
    "validation_average_precision", ascending=False
)
print(results_df.drop(columns="setting").to_string(index=False))

# Select on validation only, then generate test predictions once for the selected setting.
best_result = max(hyperparameter_results, key=lambda result: result["validation_average_precision"])
selected_setting = best_result["setting"]
selected_model = joblib.load(os.path.join(hyperparameter_results_dir, f"model_{selected_setting}.pkl"))
test_scores = selected_model.predict_proba(x_test[full_features])[:, 1]
pd.read_pickle(os.path.join(hyperparameter_results_dir, f"{selected_setting}_validation.pkl")).to_pickle(
    os.path.join(hyperparameter_results_dir, "selected_validation.pkl")
)
pd.DataFrame({"isFraud": y_test, "score": test_scores}).to_pickle(
    os.path.join(hyperparameter_results_dir, "selected_test.pkl")
)
print(f"Validation-selected min_samples_leaf: {best_result['min_samples_leaf']}")

## 4. Hyperparameter ablation: number of trees

Vary only `n_estimators` while keeping the full feature set and all other Random Forest settings fixed. Compare validation average precision (AP), select the tree count using validation only, and reserve the test set for that selected configuration.

In [5]:
from sklearn.metrics import average_precision_score

# Keep the complete feature set and all other Random Forest settings fixed.
full_features = list(x_train.columns)
n_estimators_values = [50, 100, 200, 400, 800]
n_estimators_results = []
n_estimators_results_dir = os.path.join(RESULTS_DIR, "hyperparameter_ablation_n_estimators")
os.makedirs(n_estimators_results_dir, exist_ok=True)

for tree_count in n_estimators_values:
    model = RandomForestClassifier(
        n_estimators=tree_count,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        max_samples=0.6,
        oob_score=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(x_train[full_features], y_train)

    val_scores = model.predict_proba(x_val[full_features])[:, 1]
    artifact_name = f"n_estimators_{tree_count}"
    joblib.dump(model, os.path.join(n_estimators_results_dir, f"model_{artifact_name}.pkl"))
    pd.DataFrame({"isFraud": y_val, "score": val_scores}).to_pickle(
        os.path.join(n_estimators_results_dir, f"{artifact_name}_validation.pkl")
    )
    n_estimators_results.append({
        "setting": artifact_name,
        "n_estimators": tree_count,
        "validation_average_precision": average_precision_score(y_val, val_scores),
        "oob_score": model.oob_score_,
    })

print(pd.DataFrame(n_estimators_results).drop(columns="setting").sort_values(
    "validation_average_precision", ascending=False
).to_string(index=False))

# Select using validation AP, then generate test predictions only for the selected model.
best_result = max(n_estimators_results, key=lambda result: result["validation_average_precision"])
selected_setting = best_result["setting"]
selected_model = joblib.load(os.path.join(n_estimators_results_dir, f"model_{selected_setting}.pkl"))
test_scores = selected_model.predict_proba(x_test[full_features])[:, 1]
pd.read_pickle(os.path.join(n_estimators_results_dir, f"{selected_setting}_validation.pkl")).to_pickle(
    os.path.join(n_estimators_results_dir, "selected_validation.pkl")
)
pd.DataFrame({"isFraud": y_test, "score": test_scores}).to_pickle(
    os.path.join(n_estimators_results_dir, "selected_test.pkl")
)
print(f"Validation-selected n_estimators: {best_result['n_estimators']}")

d:\AIEnvs\master_ai_env\Lib\site-packages\sklearn\ensemble\_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(
d:\AIEnvs\master_ai_env\Lib\site-packages\sklearn\ensemble\_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(
d:\AIEnvs\master_ai_env\Lib\site-packages\sklearn\ensemble\_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


 n_estimators  validation_average_precision  oob_score
          100                      0.350586   0.771808
          200                      0.350580   0.778590
          800                      0.350112   0.784318
          400                      0.349190   0.782735
           50                      0.348879   0.761211
Validation-selected n_estimators: 100


## 5. Train model with best parameters + feature ablation

In [ ]:
# # Create the results folder if it does not exist yet.
# os.makedirs(RESULTS_DIR, exist_ok=True)

# for feature_set_name, selected_features in feature_sets.items():
#     print("\nTraining Random Forest with:", feature_set_name)

#     model = RandomForestClassifier(
#         n_estimators=200,
#         max_depth=None,
#         min_samples_split=10,
#         min_samples_leaf=4,
#         max_features="sqrt",
#         max_samples=0.6,
#         oob_score=True,
#         class_weight="balanced",
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#     )
#     print(model)
#     model.fit(x_train[selected_features], y_train)

#     val_scores = model.predict_proba(x_val[selected_features])[:, 1]
#     test_scores = model.predict_proba(x_test[selected_features])[:, 1]

#     # Save the model and scores for the evaluation notebook.
#     joblib.dump(model, os.path.join(RESULTS_DIR, f"model_{feature_set_name}.pkl"))
#     pd.DataFrame({"isFraud": y_val, "score": val_scores}).to_pickle(
#         os.path.join(RESULTS_DIR, f"{feature_set_name}_validation.pkl")
#     )
#     pd.DataFrame({"isFraud": y_test, "score": test_scores}).to_pickle(
#         os.path.join(RESULTS_DIR, f"{feature_set_name}_test.pkl")
#     )

#     print("OOB score:", round(model.oob_score_, 4))
#     print("Saved model and validation/test scores.")
